# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

## Senior Technical Mentor Chatbot

In [ ]:
import json
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
#setup llm
load_dotenv(override=True)

api_key = os.getenv("GOOGLE_API_KEY")
MODEL = "gemini-3.1-flash-lite"

gemini = OpenAI(api_key=api_key, base_url = "https://generativelanguage.googleapis.com/v1beta/openai/")

In [ ]:
SYSTEM_PROMPT = """
Anda adalah Senior Technical AI & Software Engineering Mentor.
Tugas Anda adalah menjawab pertanyaan teknis pengguna dengan sangat jelas, terstruktur, 
dan praktis. Selalu berikan contoh kode jika relevan, dan jelaskan konsep sulit 
dengan analogi sederhana.
"""

In [ ]:
#define tool
def calculate_api_cost(tokens: int, cost_per_1k: float = 0.00015) -> str:
    """Hitung estimasi biaya API berdasarkan jumlah token."""
    print("Tools digunakan")
    cost = (tokens / 1000) * cost_per_1k
    return f"Estimasi biaya API untuk {tokens} token adalah: ${cost:.6f} USD"

tools_schema = [
    {
        "type": "function",
        "function":{
            "name": "calculate_api_cost",
            "description": "Hitung estimasi biaya API berdasarkan jumlah token.",
            "parameters":{
                "type": "object",
                "properties": {
                    "tokens": {
                        "type": "integer",
                        "description": "Jumlah token yang digunakan."
                    },
                    "cost_per_1k": {
                        "type": "number",
                        "description": "Biaya per 1000 token (default: $0.00015)."
                    }
                }, "required": ["tokens"]
            }
        }
    }
]

In [ ]:
def respond(message, history, model_name):
    # Pastikan message bertipe string (mencegah error 'files.get' pada objek multimodal)
    user_text = message["text"] if isinstance(message, dict) else str(message)

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    # Ekstrak history berformat tuple [user, assistant]
    for item in history:
        if isinstance(item, (list, tuple)):
            u, a = item[0], item[1]
            u_text = u["text"] if isinstance(u, dict) else str(u) if u else ""
            a_text = a["text"] if isinstance(a, dict) else str(a) if a else ""
            if u_text:
                messages.append({"role": "user", "content": u_text})
            if a_text:
                messages.append({"role": "assistant", "content": a_text})
        elif isinstance(item, dict):
            messages.append({"role": item.get("role", "user"), "content": item.get("content", "")})

    # Tambahkan input user saat ini
    messages.append({"role": "user", "content": user_text})

    # Panggilan API Pertama (Tool Check)
    response = gemini.chat.completions.create(
        model=model_name,
        messages=messages,
        tools=tools_schema,
        tool_choice="auto"
    )

    response_message = response.choices[0].message

    # Eksekusi jika LLM memanggil Tool
    if response_message.tool_calls:
        tool_call = response_message.tool_calls[0]
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        if function_name == "calculate_api_cost":
            tool_output = calculate_api_cost(
                tokens=function_args.get("tokens"),
                cost_per_1k=function_args.get("cost_per_1k", 0.00015)
            )

        messages.append(response_message)
        messages.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": tool_output,
        })

        stream = gemini.chat.completions.create(
            model=model_name,
            messages=messages,
            stream=True
        )
        partial_message = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                partial_message += chunk.choices[0].delta.content
                yield partial_message

    # Streaming teks biasa
    else:
        stream = gemini.chat.completions.create(
            model=model_name,
            messages=messages,
            stream=True
        )
        partial_message = ""
        for chunk in stream:
            if chunk.choices[0].delta.content is not None:
                partial_message += chunk.choices[0].delta.content
                yield partial_message

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 Technical Q&A Assistant")

    model_dropdown = gr.Dropdown(
        choices=["gemini-3.1-flash-lite", "gemini-3.6-flash"],
        value="gemini-3.1-flash-lite",
        label="Pilih Model LLM",
        interactive=True
    )

    chatbot = gr.ChatInterface(
        fn=respond,
        type="tuples",
        additional_inputs=[model_dropdown],
        examples=[
            ["Jelaskan apa itu Vector Embedding dalam 2 kalimat!", "gemini-3.1-flash-lite"],
            ["Berapa estimasi biaya API jika saya memproses 50000 token?", "gemini-3.1-flash-lite"],
            ["Bagaimana cara kerja Self-Attention pada Transformer?", "gemini-3.6-flash"]
        ]
    )

demo.launch()